In [ ]:
# =============================================================================
# explore/Explore_exposure_sources.py
# What do the governed exposure sources actually contain?  Read only. Writes nothing. Safe to run any time.
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Paths come from the DASH data catalogue.
#
# For each source it lists the files, then reads the schema, the row count and three sample rows (geometry left out),
# and for flood layers the distinct values of any band, probability, depth or zone column.
# It also lists the AddressBase Premium classification codes (this is how critical sites will be found) and the
# NSUL columns (UPRN, grid reference, output area, local authority, police force and so on).
# Photograph or copy the result tables back. Nothing here is secret: it is names, counts and codes.
#
# Widgets: extra_paths (more folders, comma separated), max_parts (parquet folders to read per source), count_rows (yes/no).
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql import functions as F

import config
from utils.helpers import get_spark

spark = get_spark()


def show(df):
    """Display a table with every column as text: mixed numbers and blanks cannot be converted for display."""
    display(df.fillna("").astype(str) if len(df) else df)


dbutils.widgets.text("extra_paths", "", "Extra folders to survey (comma separated)")
dbutils.widgets.text("max_parts", "6", "Parquet folders to read per source")
dbutils.widgets.dropdown("count_rows", "yes", ["yes", "no"], "Count rows")
extra = [p.strip() for p in dbutils.widgets.get("extra_paths").split(",") if p.strip()]
MAX_PARTS = int(dbutils.widgets.get("max_parts") or 6)
COUNT_ROWS = dbutils.widgets.get("count_rows") == "yes"

DEFRA = "/Volumes/prd_dash_bronze/defra_data_services_platform_unrestricted"
ONS = "/Volumes/prd_dash_bronze/ons_open_geography_portal_unrestricted"
OS_R = "/Volumes/prd_dash_bronze/ordnance_survey_data_hub_restricted"
OS_U = "/Volumes/prd_dash_bronze/ordnance_survey_data_hub_unrestricted"

SOURCES = {
    "rofrs_rivers_sea": f"{DEFRA}/rof_rivers_sea/format_GEOPARQUET_rof_rivers_sea/LATEST_rof_rivers_sea/",
    "rofsw_surface_water": f"{DEFRA}/rof_surface_water/format_GEOPARQUET_rof_surface_water/LATEST_rof_surface_water/",
    "rofsw_hazard": f"{DEFRA}/rof_sw_hazard/format_GEOPARQUET_rof_sw_hazard/LATEST_rof_sw_hazard/",
    "flood_zones": f"{DEFRA}/ea_flood_map_for_planning_flood_zones/format_GEOPARQUET_ea_flood_map_for_planning_flood_zones/LATEST_ea_flood_map_for_planning_flood_zones/",
    "flood_risk_areas": f"{DEFRA}/flood_risk_areas/format_GEOPARQUET_flood_risk_areas/LATEST_flood_risk_areas/",
    "nsul": f"{ONS}/national_statistics_uprn_lookup/format_PARQUET_national_statistics_uprn_lookup/LATEST_national_statistics_uprn_lookup/",
    "nsul_reference": f"{ONS}/national_statistics_uprn_lookup_ref/format_CSV_national_statistics_uprn_lookup_ref/LATEST_national_statistics_uprn_lookup_ref/",
    "onspd": f"{ONS}/ons_postcode_directory/format_PARQUET_ons_postcode_directory/LATEST_ons_postcode_directory/",
    "open_linked_identifiers": f"{OS_U}/open_linked_identifiers/format_PARQUET_open_linked_identifiers/LATEST_open_linked_identifiers/",
    "addressbase_premium": f"{OS_R}/addressbase_premium/format_GEOPARQUET_addressbase_premium/LATEST_addressbase_premium/",
    "local_authority_districts": f"{ONS}/local_authority_districts_bdy_uk_bfe/format_GEOPARQUET_local_authority_districts_bdy_uk_bfe/LATEST_local_authority_districts_bdy_uk_bfe/",
    "counties_unitary": f"{ONS}/counties_uni_authorities_bdy_uk_bfe/format_GEOPARQUET_counties_uni_authorities_bdy_uk_bfe/LATEST_counties_uni_authorities_bdy_uk_bfe/",
    "output_areas_2021": f"{ONS}/output_areas_2021_ew_bfe/format_GEOPARQUET_output_areas_2021_ew_bfe/LATEST_output_areas_2021_ew_bfe",
}
for i, p in enumerate(extra, 1):
    SOURCES[f"extra_{i}"] = p
print(f"{len(SOURCES)} sources to survey.")


In [ ]:
# =============================================================================
# 1. FILES: what is in each folder (two levels deep), with sizes
# =============================================================================

def list_dir(path, depth=2):
    out = []
    try:
        for f in dbutils.fs.ls(path):
            out.append({"path": f.path.replace("dbfs:", ""), "size_mb": round(f.size / 1e6, 2), "is_dir": f.name.endswith("/")})
            if f.name.endswith("/") and depth > 1:
                out.extend(list_dir(f.path, depth - 1))
    except Exception as e:
        out.append({"path": path, "size_mb": None, "is_dir": None, "error": str(e)[:100]})
    return out


rows = []
for name, path in SOURCES.items():
    listing = list_dir(path)
    files = [r for r in listing if r.get("is_dir") is False]
    errors = [r for r in listing if r.get("error")]
    rows.append({"source": name, "entries": len(listing), "files": len(files),
                 "total_mb": round(sum(r["size_mb"] or 0 for r in files), 1),
                 "first_entries": " | ".join(os.path.basename(r["path"].rstrip("/")) for r in listing[:6]),
                 "error": errors[0]["error"] if errors else ""})
inventory = pd.DataFrame(rows)
print("1. Folder inventory")
show(inventory)


In [ ]:
# =============================================================================
# 2. SCHEMAS: columns, row counts and three sample rows (geometry left out)
# A "part" is a folder that directly holds parquet files (a layer or a table).
# =============================================================================

def find_parts(path, depth=3):
    """Folders (or single files) holding parquet data, searched a few levels down."""
    try:
        entries = dbutils.fs.ls(path)
    except Exception:
        return []
    names = [e.name for e in entries]
    if any(n.endswith(".parquet") for n in names):
        return [path]
    found = []
    for e in entries:
        if e.name.endswith("/") and depth > 1:
            found.extend(find_parts(e.path, depth - 1))
    return found


def is_geometry(field):
    return field.name.lower() in ("geometry", "geom", "shape", "wkb_geometry") or field.dataType.simpleString() == "binary"


schema_rows, sample_rows, parts_read = [], [], {}
for name, path in SOURCES.items():
    if name == "nsul_reference":
        continue
    parts = find_parts(path)
    parts_read[name] = parts
    for part in parts[:MAX_PARTS]:
        label = f"{name}/{os.path.basename(part.rstrip('/'))}"
        try:
            df = spark.read.parquet(part)
            cols = [f for f in df.schema.fields]
            n = df.count() if COUNT_ROWS else None
            schema_rows.append({"part": label, "rows": n, "columns": len(cols),
                                "column_names": ", ".join(f"{f.name}:{f.dataType.simpleString()}" for f in cols)})
            keep = [f.name for f in cols if not is_geometry(f)]
            for r in df.select(*keep).limit(3).toPandas().to_dict("records"):
                sample_rows.append({"part": label, **{k: str(v)[:40] for k, v in r.items()}})
        except Exception as e:
            schema_rows.append({"part": label, "rows": None, "columns": None, "column_names": "ERROR: " + str(e)[:150]})
    if len(parts) > MAX_PARTS:
        schema_rows.append({"part": f"{name}/...", "rows": None, "columns": None, "column_names": f"{len(parts) - MAX_PARTS} more parts not read (raise max_parts)"})

print("2a. Schemas")
show(pd.DataFrame(schema_rows))
print("\n2b. Sample rows (geometry omitted)")
show(pd.DataFrame(sample_rows))


In [ ]:
# =============================================================================
# 3. BANDS: distinct values of any band, probability, depth, hazard or zone column in the flood layers
# =============================================================================

BAND_WORDS = ("band", "prob", "risk", "depth", "hazard", "zone", "category", "class", "likelihood", "thresh")
band_rows = []
for name in ("rofrs_rivers_sea", "rofsw_surface_water", "rofsw_hazard", "flood_zones", "flood_risk_areas"):
    for part in parts_read.get(name, [])[:MAX_PARTS]:
        label = f"{name}/{os.path.basename(part.rstrip('/'))}"
        try:
            df = spark.read.parquet(part)
            for f in df.schema.fields:
                if is_geometry(f) or not any(w in f.name.lower() for w in BAND_WORDS):
                    continue
                vals = [r[0] for r in df.select(f.name).distinct().limit(25).collect()]
                band_rows.append({"part": label, "column": f.name, "type": f.dataType.simpleString(),
                                  "distinct_shown": len(vals), "values": " | ".join(str(v) for v in sorted(vals, key=str))[:300]})
        except Exception as e:
            band_rows.append({"part": label, "column": "ERROR", "values": str(e)[:150]})
print("3. Band and zone columns in the flood layers")
show(pd.DataFrame(band_rows))


In [ ]:
# =============================================================================
# 4. ADDRESSBASE PREMIUM CLASSIFICATION CODES
# This is how critical sites (hospitals, schools, emergency services, utilities, prisons) can be found without
# an external list. The table name is found from the folder listing, so the code does not assume it.
# =============================================================================

ab_parts = parts_read.get("addressbase_premium", [])
print("AddressBase parts found:", [os.path.basename(p.rstrip("/")) for p in ab_parts])
class_part = next((p for p in ab_parts if "classification" in p.lower()), None)
if class_part is None:
    print("No classification folder found. Send me the folder list from table 1.")
else:
    c = spark.read.parquet(class_part)
    print("Columns:", c.columns)
    code_col = next((x for x in c.columns if "classification" in x.lower() and "code" in x.lower()), None) \
        or next((x for x in c.columns if "class" in x.lower()), None)
    print("Code column used:", code_col)
    counts = c.groupBy(code_col).count().toPandas().sort_values(code_col)
    print(f"{len(counts)} distinct codes. Codes that look like critical sites are flagged by the first letters CE, CM, CX, CU, CL, CC, ZU, UC.")
    counts["critical_prefix"] = counts[code_col].astype(str).str[:2].isin(["CE", "CM", "CX", "CU", "CC", "CL", "ZU", "UC"])
    show(counts[counts["critical_prefix"]])
    print("\nAll first-two-letter groups with address counts:")
    grp = counts.assign(prefix=counts[code_col].astype(str).str[:2]).groupby("prefix")["count"].sum().reset_index()
    show(grp)


In [ ]:
# =============================================================================
# 5. NSUL: what does one UPRN row carry?  (grid reference, output area, local authority, police force ...)
# =============================================================================

nsul_parts = parts_read.get("nsul", [])
if nsul_parts:
    n = spark.read.parquet(nsul_parts[0])
    print("NSUL columns:", ", ".join(n.columns))
    wanted = [c for c in n.columns if any(w in c.lower() for w in ("uprn", "grid", "oa21", "lsoa21", "laua", "pfa", "cauth", "fire", "lrf", "ctry", "rgn", "pcon", "imd", "ruc21"))]
    print("Columns of interest:", ", ".join(wanted))
    for c in wanted:
        if c.lower() in ("uprn", "gridgb1e", "gridgb1n"):
            continue
        k = n.select(c).distinct().count()
        print(f"  {c}: {k} distinct values")
print("\nDone. Send the tables back: inventory, schemas, sample rows, bands, classification codes.")


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")
